# Lab 05 · ReAct: choose a tool, then decide

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · 15–20 minute lab

## Let the model choose the next step

**So far:** in Lab 04, the model chose one tool for one request, and our code checked and ran it.

**New:** an *agent loop*. The model chooses an action, our code runs it, and the result (the *observation*) goes into the next prompt as part of the *history*. This act-then-observe cycle is the core of the ReAct pattern.

Our code stays in control. It runs only known tools, refuses to finish before any project facts were read, and stops after five turns. The tools are plain Python functions, as in Lab 04, so the loop is easy to follow; self-study Lab 10 serves tools like these through MCP.

**At a glance**

![Lab 05 at a glance: Goal, then Next action (JSON), then Guards (known tool · facts read), then Run the tool (list / get_project), then Observation (added to history). Then back to Next action: repeat · at most 5 turns.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/05.svg)

## The idea in context

In Lab 04 the model chose one tool for one request. Many tasks need several steps, in an order that depends on what each step finds: look up the exact name, then read the facts, then decide. ReAct (Yao and colleagues, 2022) interleaves reasoning and acting: the model picks an action, the program runs it, and the result, the observation, goes back to the model for its next choice. This loop is what most people mean by an agent.

Autonomy brings new risks: an agent can loop forever, act on a guess, or call something it should not. So the program keeps control of the loop: a turn limit, a fixed list of tools, and rules such as "read the facts before you finish". The agent decides what to do; the code decides what is allowed.

| Key term | Meaning |
| --- | --- |
| **Agent** | A system in which a model chooses its next action in a loop, based on what it has observed. |
| **Observation** | The result of an action, fed back into the next prompt. |
| **History** | Every action and observation so far; the agent's view of the task. |
| **Guard** | A rule in code that the agent cannot override, such as a turn limit. |

## Pattern · One-Step Tool Agent

*The same single-agent pattern as Lab 04, from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide; see Lab 04 for the full card. Related: Incremental Tool Agent, Registry & Adapter (Lab 10).*

| | |
| --- | --- |
| **What changes** | Lab 04 made one tool call per request. Here each turn is one One-Step Tool Agent call, and repeating it, with every observation in the history, makes a *ReAct* loop. |
| **Flow** | Goal → Choose action → Tool → Observe → repeat, or Finish |
| **New liability** | A loop can run forever, or finish before it has looked at anything. The turn limit and the finish guard address both, on top of Lab 04's checks of each request. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · The tools the agent may use

`list_projects` returns the exact project names. `get_project` returns one project's facts, or an error message the model can read and recover from. `TOOLS` maps each action name to its function, so the loop can look up whatever the model asks for.

In [ ]:
import json

PROJECTS = {
    "Course portal":     {"team_size": 3,  "independent_deploys": False},
    "Payments platform": {"team_size": 12, "independent_deploys": True},
    "Internal wiki":     {"team_size": 9,  "independent_deploys": False},
}

def list_projects(_):
    return sorted(PROJECTS)

def get_project(name):
    return PROJECTS.get(name, f"Unknown project {name!r}. Call list_projects for exact names.")

TOOLS = {"list_projects": list_projects, "get_project": get_project}
print(TOOLS["get_project"]("wiki"))

## Step 2 · Ask the model for its next action

Each prompt holds the policy, the goal, the tools, and the whole history so far. The model replies with one JSON action: a tool and its input, or `finish` with the decision.

The goal says "wiki", not the exact name "Internal wiki", so the model has to look up the name first. In practice mode, `script` holds one scripted reply per turn. `next(script, "")` takes the next one, or an empty reply when the script runs out. A live model may wrap its JSON in extra text; then `json.loads` raises an error, which is the safe outcome. Lab 01's parser shows how to report such replies.

In [ ]:
POLICY = "Choose microservices only if team_size >= 8 AND independent_deploys is true; otherwise modular monolith."
goal = "Decide the architecture for our wiki project."
script = iter(['{"action": "list_projects", "input": ""}',
               '{"action": "get_project", "input": "Internal wiki"}',
               '{"action": "finish", "input": "modular monolith"}'])

def next_action(history):
    """Show the model the goal and everything observed so far; return its JSON action."""
    prompt = (f"{POLICY}\nGoal: {goal}\n"
              "Tools: list_projects (input ignored), get_project (input: exact project name)\n"
              f"History: {json.dumps(history)}\n"
              'Reply with JSON only, no code fences: {"action": "<tool name or finish>", "input": "<argument or decision>"}')
    return json.loads(ask(prompt, example=next(script, "")))

## Step 3 · Run the loop with guards

Each turn either finishes or runs one tool and records what it observed. Three guards keep the agent safe:

- An unknown action does not crash the loop. It becomes an observation that tells the model what went wrong.
- `finish` is refused until a `get_project` call has read real facts.
- `range(1, 6)` allows at most five turns, so the loop always ends.

In [ ]:
history, decision = [], None
for turn in range(1, 6):
    step = next_action(history)
    action, value = step["action"], step["input"]
    print(f"Turn {turn}: {action}({value!r})")
    if action == "finish":
        if not any(h["action"] == "get_project" for h in history):
            raise ValueError("The model tried to finish before reading any project facts.")
        decision = value.strip().lower()
        break
    tool = TOOLS.get(action)
    observation = tool(value) if tool else f"Unknown action {action!r}. Use one of {list(TOOLS)}."
    history.append({"action": action, "input": value, "observation": observation})
    print("        observed:", observation)
assert decision is not None, "The turn limit was reached before the model finished."

## Step 4 · Check the agent's decision

The agent's answer is still only a claim. Like a test in Lab 01, the check knows which project the goal means, and applies the policy to that project's facts. Counting the tool calls shows the cost of the run.

In [ ]:
facts = PROJECTS["Internal wiki"]  # the project the goal refers to
expected = "microservices" if facts["team_size"] >= 8 and facts["independent_deploys"] else "modular monolith"
print(f"Agent decided: {decision} | policy expects: {expected} | tool calls: {len(history)}")
assert decision == expected, "The agent's decision breaks the policy."

## What you should see

Step 1 prints the error message for the unknown name `wiki`. Step 3 shows three turns: `list_projects`, then `get_project('Internal wiki')` with its facts, then `finish('modular monolith')`. Step 4 prints `Agent decided: modular monolith | policy expects: modular monolith | tool calls: 2`.

## Try it

1. Make the agent skip the lookup: delete the first two scripted replies, leaving only `finish`. Run Steps 2 and 3. The finish guard stops the run.
2. Make the agent guess a name: change the second scripted reply's input to `"wiki"`. Its observation is now an error message. Add a fourth scripted reply that calls `get_project` with `"Internal wiki"` before `finish`, and the agent recovers.
3. Change the goal to the payments project, and update the script and the project name in Step 4. What should the decision be?
4. In live mode, how many turns does a real model take for the original goal?

## Takeaway

An agent is a loop: decide, act, observe, repeat. The model chooses the steps, but our code owns the tools, the guards, and the turn limit, and it checks the result at the end. Lecture 3 starts with Lab 06, which puts three architects, each driven by its own model, into one council.

## Read further

- [ReAct](https://arxiv.org/abs/2210.03629)
- [Workflows and agentic patterns](https://www.anthropic.com/research/building-effective-agents)